In [44]:
# 1) ติดตั้ง (ครั้งแรกใน Colab เท่านั้น)
!pip install -q pandas scikit-learn joblib openpyxl

In [55]:
# 2) โหลดไฟล์ (หรือ DataFrame ที่สร้างเอง)
#    • ถ้าคุณเซฟ mock ไว้เป็น .csv แล้ว:
#      df = pd.read_csv('/content/fact_eta_10k_full.csv')
#    • ถ้าอยู่ใน Excel sheet:
import pandas as pd
df = pd.read_excel('/content/sample_data/data.xlsx', sheet_name='Sheet4')

print(df)

       order_id shop_guid  app_source                  order_time  \
0     ORD400000    SHOP_J   Robinhood  2025-04-27 09:58:01.702287   
1     ORD400001    SHOP_I   Robinhood  2025-04-27 11:07:36.650238   
2     ORD400002    SHOP_D  ShopeeFood  2025-04-28 05:19:30.056918   
3     ORD400003    SHOP_I    GrabFood  2025-04-27 22:09:09.361013   
4     ORD400004    SHOP_C    GrabFood  2025-04-28 05:23:51.666059   
...         ...       ...         ...                         ...   
9995  ORD409995    SHOP_G   Robinhood  2025-04-27 22:19:53.741994   
9996  ORD409996    SHOP_J  ShopeeFood  2025-04-27 10:03:25.673139   
9997  ORD409997    SHOP_I   Robinhood  2025-04-27 20:00:41.220213   
9998  ORD409998    SHOP_B    LINE_MAN  2025-04-27 09:43:04.833635   
9999  ORD409999    SHOP_F    LINE_MAN  2025-04-28 04:43:56.669318   

      distance_km  queue_size  prep_time_sec  traffic_sec  rain_flag  \
0            1.38           2            935          281          0   
1            1.58          

In [56]:
# 3) (กรณีไฟล์ยังไม่มี prep_ahead_sum_sec)  ➜ คำนวณเพิ่ม
#    สมมุติ df มีคอลัมน์ queue_size_total และ prep_time_sec ของทุกใบในคิว
#    ทางง่าย: สุ่มจำลองเหมือน mock ตัวอย่าง
import numpy as np
np.random.seed(2025)
df["prep_ahead_sum_sec"] = [
    np.random.randint(240, 1201, q).sum() if q > 0 else 0
    for q in df["queue_size"]  # หรือ queue_size_total
]

In [57]:
# 4) แยกฟีเจอร์ / Target  (เพิ่ม prep_ahead_sum_sec เข้าไป)
# -------------------------------------------------
FEATURES = ['distance_km',
            'queue_size',          # หรือ queue_size_total
            'prep_ahead_sum_sec',  # ⭐ ใหม่
            'prep_time_sec',
            'traffic_sec',
            'rain_flag'            # ใส่/ตัดได้ตามจริง
]
X = df[FEATURES]
y = df['eta_actual_min']
print(X)
print(y)


      distance_km  queue_size  prep_ahead_sum_sec  prep_time_sec  traffic_sec  \
0            1.38           2                1788            935          281   
1            1.58           1                1102            321          189   
2            2.68          10                6883            342          321   
3            4.61           2                1551            793          719   
4            3.52          10                6388            978          718   
...           ...         ...                 ...            ...          ...   
9995         8.61           3                1821            424         1033   
9996         1.31           9                5629            575          267   
9997         1.67           1                 465            516          200   
9998         8.54           4                2495           1039         1742   
9999         7.00           4                2944           1104         1092   

      rain_flag  
0        

In [58]:
# 5) Train / Test split  &  Fit model
# -------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

model = LinearRegression().fit(X_train, y_train)

In [59]:
# 6) ประเมินความแม่น
# -------------------------------------------------
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error # Import root_mean_squared_error
y_pred = model.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
# rmse = mean_squared_error(y_test, y_pred, squared=False) # Remove this line
rmse = root_mean_squared_error(y_test, y_pred) # Calculate RMSE using root_mean_squared_error
print(f"MAE  : {mae:.2f} นาที")
print(f"RMSE : {rmse:.2f} นาที")

MAE  : 2.38 นาที
RMSE : 3.00 นาที


In [60]:
# 7) บันทึกโมเดล
# -------------------------------------------------
import joblib
joblib.dump(model, 'eta_model_with_prepAhead.pkl')

# -------------------------------------------------
# 8) ใช้โมเดลทำนายใบใหม่ (ต้องมี prep_ahead_sum_sec เสมอ)
# -------------------------------------------------
new_order = pd.DataFrame([{
    'distance_km':  3.2,
    'queue_size':   4,
    'prep_ahead_sum_sec': 5400,   # ต้องคำนวณจากคิวจริง
    'prep_time_sec': 780,
    'traffic_sec':   900,
    'rain_flag':     1
}])
print("ETA คาด ≈",
      joblib.load('eta_model_with_prepAhead.pkl').predict(new_order)[0])

ETA คาด ≈ 35.97132555055097


In [67]:
# ---------- 1) บันทึกค่าพยากรณ์ ----------
import pandas as pd
import joblib

# Assuming 'test_order' and 'loaded' are defined
test_order = pd.DataFrame([{
    'distance_km': 3.2,
    'queue_size': 4,
    'prep_ahead_sum_sec': 5400,
    'prep_time_sec': 780,
    'traffic_sec': 900,
    'rain_flag': 1
}])
loaded = joblib.load('eta_model_with_prepAhead.pkl')

pred_log = pd.DataFrame([{
    'order_id': 'XT123',
    'order_time': pd.Timestamp('2025-04-27 12:34:00'),
    'eta_pred_min': loaded.predict(test_order)[0]
}])

# ---------- 2) สมมุติว่าได้รับเวลาส่งจริง ----------
delivered_ts = pd.Timestamp('2025-04-27 13:02:00')  # สมมุติ

# ---------- 3) คำนวณ ETA จริง & Error ----------
pred_log['delivered_ts'] = delivered_ts
pred_log['eta_actual_min'] = (delivered_ts - pred_log['order_time']).dt.total_seconds() / 60
pred_log['error_min'] = abs(pred_log['eta_actual_min'] - pred_log['eta_pred_min'])

print(pred_log[['eta_pred_min', 'eta_actual_min', 'error_min']])

   eta_pred_min  eta_actual_min  error_min
0     35.971326            28.0   7.971326


# ส่วนใหม่